[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/articulated/articulated.ipynb)

# Articulated Robot Dynamics

This example computes the dynamics of an articulated robot in reduced coordinates: Brax's humanoid, a torso with a waist, two legs and two arms, thirteen bodies turning on seventeen hinges, whose state is the torso's placement and rate and the hinges' angles and speeds. Inverse dynamics finds the forces that give the torso and the hinges given accelerations; forward dynamics finds the accelerations that given forces produce, by solving the mass matrix, which says how hard the torso and each hinge are to accelerate and how they drag on each other. Dropped onto level ground with its hinges as limp as Brax's weak springs make them, the humanoid lands on its feet, its knees buckle, and it slumps into a heap.

In projective geometric algebra a hinge's axis and a link's rate are lines to turn about, bivectors, while a link's momentum and the forques on it are lines joined from points, antibivectors of the opposite orientation. One motor sandwich moves each of them between frames, `motor >> axis` and `motor >> forque`, and moves a link's inertia, a map from rates to forques, as `motor >> inertia(motor << Rate)`. The two cross products of rigid-body dynamics, one for motions and one for forces, are one commutator: a hinge's axis carried by a moving link changes at `rate.commutator(axis)`, and a momentum carried by its motion at `rate.commutator(momentum)`, the operands' types keeping motions and forces apart. Each link's rate is the torso's rate plus the hinges' speeds carried out along its path from the torso, by a sparse map whose cells are the hinges' axes; its adjugate carries the links' forques back, pairing each hinge's axis with the forque on the subtree beyond it, `axis & forque`, while the torso bears the sum of them all. Recursive Newton–Euler inverse dynamics is that map forward and its adjugate back, and the mass matrix is the links' inertias pulled back through it, `joint_rates.adjugate()(inertia(joint_rates))`, symmetric and as sparse as the tree by construction. The ground's push where it meets one of the balls that end Brax's capsules, or one of its spheres, is a forque like any other, and over an implicit step its damping and stiffness, joined with the ball's lowest point as an inertia joins its points with their velocities, are extra inertia on the ball's link, pulled back the same way. JAX compiles the step, which moves a whole batch of humanoids in one call.

In Featherstone's spatial vector notation, motions and forces are six-vectors that change frame by two matrices, ${}^B X_A$ for motions and ${}^B X_A^* = {}^B X_A^{-T}$ for forces, and a spatial inertia changes as $I_B = {}^B X_A^* \, I_A \, {}^A X_B$. Motions have the cross product $\mathrm{crm}(v) = v\times$ and forces $\mathrm{crf}(v) = v\times^* = -\mathrm{crm}(v)^T$. The recursive Newton–Euler algorithm runs $v_i = v_{\lambda(i)} + S_i \dot q_i$ and $a_i = a_{\lambda(i)} + S_i \ddot q_i + v_i \times S_i \dot q_i$ outward from the root, forms $f_i = I_i a_i + v_i \times^* I_i v_i - f_i^x$ with the external forces $f_i^x$, and runs $\tau_i = S_i^T f_i$ and $f_{\lambda(i)} \mathrel{+}= f_i$ back in; a floating base is a joint with $S_0 = \mathbf{1}$. The joint-space inertia is $H = S^T I^C S$, entry by entry $H_{ij} = S_i^T I^C_i S_j$ for $j$ on the path to $i$, from the composite inertia $I^C_i$ of each subtree.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from functools import partial

import jax
import numpy as np

from numga import Extensor, stack, where
from numga.algebras import PGA3D as ga
from numga.backend.jax import JaxContext
from numga.sparse import SparseExtensor, spdiag
from examples.mechanics.articulated import render, scenarios
from examples.mechanics.spinning_top import core as contact
from examples.util.animation import show_states

jax.config.update("jax_enable_x64", True)
mv = JaxContext(ga, dtype=np.float64).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Point = ga.gatype.antivector()                                               # Antivector-
# A joint's axis per unit speed, and a link's rate: a line to turn about, scaled.
Rate = ga.gatype.bivector()                                                  # Bivector+
# A link's momentum and the forques on it: lines joined from points.
Forque = ga.gatype.antibivector()                                            # Antibivector-
Motor = ga.gatype.rotor()                                                    # Rotor+
# A joint's momentum and its generalized force: what its speed pairs with into energy and power.
Amount = ga.gatype.antiscalar()                                              # Antiscalar-
Plane = ga.gatype.vector()                                                   # Vector+
Inertia = ga.gatype((Forque, Rate))
Quadric = ga.gatype((Plane, Point))

## The humanoid and its parts

The humanoid is a tree of links turning on hinges, rooted at the torso. Where a body turns on two or three hinges at one pivot, as the waist, a hip and a shoulder do, massless links sit between them, and each foot rides on its shin. What the dynamics needs of the tree is its paths, the hinges between the torso and each link. The parts are Brax's capsules and spheres, solid at MuJoCo's default density, each with its mass and moments carried by six samples. A part's inertia, the map from its rate to its momentum, joins every sample with the velocity the rate gives it, `samples.regressive(samples.commutator(Rate) * masses)`, and a link's is the sum of its parts', none for a massless link. A flat ground meets a capsule at one of the balls at its ends, so the balls the ground can meet are each capsule's two ends and each sphere. Forward kinematics places the torso by its motor and each other link by its parent's motor followed by its own hinge's turn about its axis in the home pose. The picture shows the humanoid as it is dropped, its torso raised and tilted.

In [ ]:
# The humanoid's tree, from its bodies and hinges: the torso, each hinge's parent and child link, the hinges on the path
# from the torso to each link, the link carrying each ball the ground can meet, and how many hinges the longest path
# passes; and the link carrying each of its parts.
(torso, tails, heads, paths, balls, n_generations), parts = scenarios.humanoid_tree()
capsules, spheres = scenarios.capsule_bodies, scenarios.sphere_bodies
# The humanoid in its home pose, its torso at the origin: each hinge's axis, six mass samples of each part, the
# capsules then the spheres, and their masses, and the balls the ground can meet and their radii.
home_axes, samples, masses, centres, radii = scenarios.home_pose()          # Rate+[joints], [parts, 6] Point-, ...
# Each part's inertia, the map from its rate to its momentum: every sample joined with its velocity, weighted; a
# link's, the sum of its parts'.
part_inertias = samples.regressive(samples.commutator(Rate) * masses).sum(axis=-1)   # [parts] Forque- <- Rate+
inertias = ~parts * part_inertias.field()                                    # (Forque- <- Rate+)[links]
# Each link's mass point, its parts' centres weighted by their masses.
mass_points = ~parts * (samples * masses).sum(axis=-1).field()               # Point-[links]


@jax.jit
def forward_kinematics(motor: Motor, angles: Scalar) -> Motor:
    """Each link's motor, `Motor[links]`, the torso placed by its motor and the hinges at their angles: its parent's
    motor, then its own hinge's turn about its axis in the home pose."""
    turns = (home_axes * (angles / 2)).exp()                                 # Motor+[joints]
    placed = ~torso * motor[..., None].field()                               # Motor+[links]
    motors = placed
    for _ in range(n_generations):
        motors = placed + ~heads * ((tails * motors) * turns)
    return motors


# The humanoid dropped: its torso raised, tilted and spinning, its hinges at rest in the home pose.
dropped = scenarios.dropped()
carried = (balls * forward_kinematics(dropped.motor, dropped.angles)).batch()   # [1, balls] Motor+
render.pose((carried >> centres)[0], radii, len(capsules));

## Inverse dynamics

Inverse dynamics finds the generalized forces that give the torso and the joints given accelerations at given rates. The joint-rate map, each joint's axis where its parent has carried it, `(tails * motors) >> home_axes`, summed along the paths, carries the joints' speeds out to the links' rates on top of the torso's. A joint's axis turns with its link, so a link's acceleration also gains the velocity product `rate.commutator(axis) * speed` of every joint on its path. Each link needs a forque that changes its momentum, at its acceleration and as its own motion turns it, `rate.commutator(momentum)`, less the forques on it from outside. Back along the paths, the torso bears the sum of all those forques, and the adjugate of the joint-rate map sums each joint's subtree and pairs it with the joint's axis, `axis & forque`, an amount, so that a joint's power is its generalized force paired with its speed.

In [ ]:
def inverse_dynamics(motors: Motor, rate: Rate, speeds: Scalar, acceleration: Rate, accelerations: Scalar,
                     external: Forque) -> tuple[Forque, Amount]:
    """The torso's forque and the joints' generalized forces, `Amount[joints]`, that give the torso and the joints their
    accelerations at their rate and speeds, under forques on the links from outside."""
    axes = (tails * motors) >> home_axes                                     # Rate+[joints]
    joint_rates = paths * spdiag(axes * Scalar)                              # (Rate+ <- Scalar+)[links <- joints]
    inertia = spdiag(motors >> inertias(motors << Rate))                     # (Forque- <- Rate+)[links <- links]
    # Out from the torso: the links' rates, and their accelerations with each joint's velocity product.
    rates = joint_rates(speeds) + rate                                       # Rate+[links]
    velocity_products = (heads * rates).commutator(axes) * speeds           # Rate+[joints]
    link_accelerations = joint_rates(accelerations) + paths * velocity_products + acceleration   # Rate+[links]
    # The forque each link needs: the change of its momentum, less what acts on it from outside.
    momenta = inertia(rates)                                                 # Forque-[links]
    forques = inertia(link_accelerations) + rates.commutator(momenta) - external   # Forque-[links]
    # Back: the torso bears them all; each joint's axis is paired with the forque on its subtree.
    return forques.sites.sum(), joint_rates.adjugate()(forques)

## The mass matrix and forward dynamics

The mass matrix is the links' inertias pulled back through the maps from the torso's rate and the joints' speeds to the links' rates, and any other map from a link's rate to a forque pulls back the same way. The torso's rate reaches every link alike, so its block is the links' maps summed, the whole robot's inertia, and its coupling to the joints is the joint-rate map's adjugate applied to them. The joints' own block, `joint_rates.adjugate()(spdiag(inertias)(joint_rates))`, is symmetric because the same map stands on both sides of the inertias, and couples two joints only where one lies on the other's path, here the hip and the knee of each leg, with no mask or mirrored triangle to fill in. A sparse map holds cells of one type, a rate per unit speed for the joints, so the torso's block stays apart and the solve eliminates it: the torso gives way to the joints' forces through the whole robot's inertia. Forward dynamics solves the mass matrix against the forces less the bias, the forces inverse dynamics gives at no acceleration.

In [ ]:
def pullback(joint_rates: SparseExtensor, link_maps: Inertia) -> tuple[Inertia, Extensor, SparseExtensor]:
    """Maps from each link's rate to a forque, `(Forque <- Rate)[links]`, pulled back in blocks: the torso's,
    `Forque <- Rate`; the joints' per rate of the torso, `Amount[joints] <- Rate`; and the joints' own,
    `(Amount <- Scalar)[joints <- joints]`."""
    return (link_maps.sites.sum(), joint_rates.adjugate()(link_maps),
            joint_rates.adjugate()(spdiag(link_maps)(joint_rates)))


def solved(whole: Inertia, coupling: Extensor, joints: SparseExtensor, forque: Forque,
           forces: Amount) -> tuple[Rate, Scalar]:
    """The torso's rate and the joints' speeds that a map in blocks takes to a forque on the torso and forces at the
    joints."""
    # The joints' block with the torso's eliminated. The sparse solve runs on SciPy alone, so under JAX each robot's
    # small block is solved dense.
    reduced = joints.to_dense() - coupling(whole.solve(coupling.adjugate()))   # (Amount- <- Scalar+)[joints <- joints]
    speeds = reduced.solve(forces - coupling(whole.solve(forque)))           # Scalar+[joints]
    return whole.solve(forque - coupling.adjugate()(speeds)), speeds


def mass_matrix(motors: Motor) -> tuple[Inertia, Extensor, SparseExtensor]:
    """The mass matrix in blocks, the links' inertias where they are pulled back."""
    axes = (tails * motors) >> home_axes                                     # Rate+[joints]
    joint_rates = paths * spdiag(axes * Scalar)                              # (Rate+ <- Scalar+)[links <- joints]
    return pullback(joint_rates, motors >> inertias(motors << Rate))


def forward_dynamics(motors: Motor, rate: Rate, speeds: Scalar, forces: Amount, external: Forque) -> tuple[Rate, Scalar]:
    """The torso's acceleration and the joints', under generalized forces at the joints and forques on the links from
    outside, a forque on the torso among them."""
    torso_bias, joint_bias = inverse_dynamics(motors, rate, speeds, rate * 0, speeds * 0, external)
    return solved(*mass_matrix(motors), -torso_bias, forces - joint_bias)

## Gravity and the ground

A link's weight is gravity joined with its mass point, `gravity & mass_points`. The ground pushes each ball's lowest point up, by a spring damped in proportion to its depth, and rubs against its slip in proportion to the push. Joined with that point as an inertia joins its points with their velocities, the push's damping and stiffness are maps from the link's rate to a forque, extra inertia on the link over a step.

In [ ]:
# Gravity downward, in metres per second squared. The ground, the plane through the origin facing up; the surfaces of
# the balls that can meet it, in the home pose; its push on each ball per depth below it, in newtons per metre, the
# share the push grows by per speed of sinking, in seconds per metre, and its friction per push and speed of slip.
gravity = -mv.z.dual() * 9.81                                                # [] Point-
ground, surfaces, contact_stiffness, contact_damping, friction = scenarios.ground()   # [] Plane+, [balls] Plane+ <- Point-, Scalar+[balls]


def ground_contact(motors: Motor, rates: Rate) -> tuple[Forque, Inertia, Inertia]:
    """The ground's forques on the links, `Forque[links]`, and how they change, by the links' rates and by their
    placement, `(Forque <- Rate)[links]` each."""
    up = ground.dual()                                                       # [] Point-
    # Each ball's surface where its link has carried it, and its lowest point against up.
    carried = (balls * motors).batch()                                       # [..., balls] Motor+
    bottoms = contact.lowest_point(carried >> surfaces(carried << Point), up).field()   # Point-[balls]
    velocities = (balls * rates).commutator(bottoms)                         # Point-[balls]
    # A direction's part along up, and how deep each lowest point lies and how fast it sinks.
    along = up * (ground & Point)                                            # [] Point- <- Point-
    depths = (-(ground & bottoms)).clip(0, np.inf)                           # Scalar+[balls]
    sinking = -(ground & velocities)                                         # Scalar+[balls]
    # The push never pulls a ball down, however fast it rises.
    pushes = (depths * contact_stiffness * (1 + contact_damping * sinking)).clip(0, np.inf)   # Scalar+[balls]
    forces = up * pushes - (velocities - along(velocities)) * (pushes * friction)   # Point-[balls]
    # How the force falls as the point moves and as it is moved, joined with the point as an inertia joins its points.
    dampers = along * (depths * contact_stiffness * contact_damping) + (Point - along) * (pushes * friction)
    # The spring's stiffness where a ball presses into the ground, none where it does not.
    springs = along * where(depths > 0, contact_stiffness, 0)                # (Point- <- Point-)[balls]
    damping = bottoms.regressive(dampers(bottoms.commutator(Rate)))          # (Forque- <- Rate+)[balls]
    stiffness = bottoms.regressive(springs(bottoms.commutator(Rate)))        # (Forque- <- Rate+)[balls]
    return ~balls * (forces & bottoms), ~balls * damping, ~balls * stiffness

## The fall

Brax's springs and dampers in the hinges pull the limbs weakly towards the home pose, and stiff one-sided springs, the stops, hold each hinge within its range, so that no knee or elbow bends backwards. A linearly implicit Euler step, at the 10 ms a robot simulator takes, solves for the change of the torso's rate and the hinges' speeds with the springs and dampers already acting over the step: its matrix is the mass matrix with the damping times the step and the stiffness times its square added, and its forces are those at the step's start, every spring as the present rates will have moved it. The hinges' and the stops' damping and stiffness add to the hinges' block, and the ground's, extra inertia on the links, go through the same pullback as the links' inertias. The velocity products stay at the step's start, as the humanoid turns slowly. The humanoid lands on its feet, its knees buckle, and it slumps into a lopsided heap; its links pass through each other, as nothing keeps them apart but the ground. `python -m examples.mechanics.articulated.scenarios` also times the compiled step for batches of up to a few thousand humanoids.

In matrix notation the step solves $(M + h D + h^2 K)\,\Delta v = h\,(f - h K v)$ with the step $h$, and the contact enters $D$ and $K$ as $J_c^T D_c J_c$ and $J_c^T K_c J_c$ through the Jacobian $J_c$ of the balls' lowest points.

In [ ]:
# Each hinge's maps from its angle and its speed to its generalized force, from Brax's weak springs and dampers; the ends
# of its range; and the stops' map there, from their stiffness.
stiffness, damping = scenarios.hinges()                                      # Scalar+[joints] each
hinge_stiffness, hinge_damping = stiffness * Scalar.dual(), damping * Scalar.dual()   # (Amount- <- Scalar+)[joints] each
low, high, stop_stiffness = scenarios.stops()                                # Scalar+[joints] each
stops = stop_stiffness * Scalar.dual()                                       # (Amount- <- Scalar+)[joints]
# The time step, in seconds, the steps a frame, the frames, and a frame's duration in milliseconds.
dt, n_steps, n_frames, frame_duration = 0.01, 4, 60, 40


def step(state: tuple) -> tuple:
    """The torso's motor and rate and the hinges' angles and speeds dt later, by a linearly implicit Euler step."""
    motor, rate, angles, speeds = state
    motors = forward_kinematics(motor, angles)
    axes = (tails * motors) >> home_axes                                     # Rate+[joints]
    joint_rates = paths * spdiag(axes * Scalar)                              # (Rate+ <- Scalar+)[links <- joints]
    rates = joint_rates(speeds) + rate                                       # Rate+[links]
    pushes, damping, stiffness = ground_contact(motors, rates)
    weights = gravity & (motors >> mass_points)                             # Forque-[links]
    # The ground's springs as the present rates will have moved them by the step's end.
    external = weights + pushes - spdiag(stiffness)(rates) * dt              # Forque-[links]
    # How far each hinge lies past its stops, and the stops' stiffness where it does, none where it does not.
    past = (angles - high).clip(0, np.inf) + (angles - low).clip(-np.inf, 0)   # Scalar+[joints]
    stopping = where(past.abs() > 0, stops, 0)                               # (Amount- <- Scalar+)[joints]
    # The hinges' springs and the stops as the present speeds will have moved them, and the dampers.
    moved = speeds * dt                                                      # Scalar+[joints]
    drive = -(spdiag(hinge_stiffness)(angles + moved) + spdiag(stopping)(past + moved) + spdiag(hinge_damping)(speeds))
    torso_bias, joint_bias = inverse_dynamics(motors, rate, speeds, rate * 0, speeds * 0, external)
    # The step's matrix: each link heavier by its balls' contact over the step, each hinge by its springs and stops.
    link_maps = (motors >> inertias(motors << Rate)) + damping * dt + stiffness * dt**2   # (Forque- <- Rate+)[links]
    whole, coupling, joints = pullback(joint_rates, link_maps)
    hinges = spdiag(hinge_damping * dt + (hinge_stiffness + stopping) * dt**2)  # (Amount- <- Scalar+)[joints <- joints]
    rate_change, speed_changes = solved(whole, coupling, joints + hinges, -torso_bias * dt, (drive - joint_bias) * dt)
    rate, speeds = rate + rate_change, speeds + speed_changes
    # The torso turns about its new rate over the step, `motor' == rate * motor / 2`.
    return (rate * (dt / 2)).exp() * motor, rate, angles + speeds * dt, speeds


@partial(jax.jit, static_argnames="n_steps")
def advance(state: tuple, n_steps: int) -> tuple:
    """The state n_steps of dt later."""
    joint_type = state[2].gatype

    def typed(state: tuple) -> tuple:
        """The state with each part of its full type, the same at every step."""
        motor, rate, angles, speeds = state
        return motor.retyped(Motor), rate.retyped(Rate), angles.retyped(joint_type), speeds.retyped(joint_type)

    return jax.lax.fori_loop(0, n_steps, lambda index, state: typed(step(state)), typed(state))


def run(state: tuple) -> Iterator[tuple]:
    """The state at every frame, from the drop on."""
    for _ in range(n_frames):
        yield state
        state = advance(state, n_steps)


def drawn(states: Iterable[tuple]) -> Iterator[np.ndarray]:
    """The frames of the states as they come, every ball where its link has carried it."""
    for motor, _, angles, _ in states:
        carried = (balls * forward_kinematics(motor, angles)).batch()       # [1, balls] Motor+
        yield from render.animate([carried >> centres], radii, len(capsules))


states = show_states(run(tuple(dropped)), drawn, name="articulated", duration_ms=frame_duration)

In [ ]:
# checks
# Forward dynamics undoes inverse dynamics, for the last frame's rate and speeds taken as accelerations.
motor, rate, angles, speeds = states[-1]
motors = forward_kinematics(motor, angles)
weights = gravity & (motors >> mass_points)                                 # Forque-[links]
forque, forces = inverse_dynamics(motors, rate, speeds, rate, speeds, weights)
# The torso's forque is one more load on the torso's link.
loaded = weights + ~torso * forque[..., None].field()                        # Forque-[links]
assert (forward_dynamics(motors, rate, speeds, forces, loaded)[1] - speeds).norm().to_array().max() < 1e-8
# The hinges' block of the mass matrix is inverse dynamics of a unit acceleration of each hinge, at rest with nothing
# acting from outside, and it is symmetric.
whole, coupling, joints = mass_matrix(motors)
units = mv.scalar(np.eye(len(scenarios.joint_names))[:, None, :, None]).field()   # [joints, 1] Scalar+[joints]
_, columns = inverse_dynamics(motors, rate * 0, units * 0, rate * 0, units, weights * 0)
entries = (joints(units)[:, None] & units[None, :]).sites.sum().to_array()   # [joints, joints, 1]
assert np.abs(entries - (columns[:, None] & units[None, :]).sites.sum().to_array()).max() < 1e-10
assert np.abs(entries - np.swapaxes(entries, 0, 1)).max() < 1e-12


@jax.jit
def energy(state: tuple) -> Scalar:
    """The kinetic energy, from the mass matrix's blocks, and the work stored against gravity, in the hinges' springs,
    in the stops past which the hinges lie and in the ground under the balls."""
    motor, rate, angles, speeds = state
    motors = forward_kinematics(motor, angles)
    whole, coupling, joints = mass_matrix(motors)
    kinetic = (whole(rate) & rate) / 2 + (coupling(rate) & speeds).sites.sum() + (joints(speeds) & speeds).sites.sum() / 2
    past = (angles - high).clip(0, np.inf) + (angles - low).clip(-np.inf, 0)   # Scalar+[joints]
    sprung = (spdiag(hinge_stiffness)(angles) & angles).sites.sum() / 2 + (spdiag(stops)(past) & past).sites.sum() / 2
    carried = (balls * motors).batch()                                       # [..., balls] Motor+
    bottoms = contact.lowest_point(carried >> surfaces(carried << Point), ground.dual()).field()   # Point-[balls]
    depths = (-(ground & bottoms)).clip(0, np.inf)                           # Scalar+[balls]
    lifted = (gravity.dual() & (motors >> mass_points)).sites.sum()
    return kinetic + lifted + sprung + (contact_stiffness * depths * depths).sites.sum() / 2


# Without a drive the step only takes energy away, but for what a ball touching down within a step stores, meeting the
# ground at the step's end: the energy ends a small share of the drop's, and no frame gains more than a few percent.
energies = stack([energy(state) for state in states])                       # [frames, 1] Scalar+
assert ((energies[1:] - energies[:-1]) / energies[0].abs()).to_array().max() < 0.03
assert (energies[-1] / energies[0]).to_array().max() < 0.25